# Functions

> ### Learning Objectives
>
> By the end of this chapter you should be able to work with:
>
> - What a function is and why it is a powerful abstraction tool (hiding detail, generalizing a process)
> - Defining and calling functions: the `def` keyword, indentation, parameters vs. arguments, and return values
> - Function scope — local variables and how scope prevents name collisions
> - Cohesion — designing functions that accomplish a single, clear task
> - Refactoring repeated code into reusable, parameterized functions
> - Generalization — moving from a fixed solution (count spaces) to a broader one (count any character)
> - Documenting functions with docstrings and type hints
> - Default arguments and keyword arguments to improve flexibility and readability
> - Function-ordering rules ("define before call") and interpreting function-related errors

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  SETUP — run this cell first.
#
#  It draws every figure used in this chapter and switches the notebook into
#  "show me every result" mode.  Everything it needs is right here: nothing to
#  install, nothing to download, no other files required.
#
#  (Curious what a figure is made of?  The drawing code is all below.)
# ══════════════════════════════════════════════════════════════════════
import io

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Polygon, Circle
from matplotlib.lines import Line2D
from IPython.display import Image, display
from IPython.core.interactiveshell import InteractiveShell

# echo the value of *every* expression in a cell, the way the Python prompt
# does — many examples in this book show several results at once
InteractiveShell.ast_node_interactivity = "all"

# ------------------------------------------------------------- drawing ---

INK = "#1a1a1a"

MUTED = "#6b7280"

FILL = "#eef2f7"

ACCENT = "#2563eb"

WARM = "#b45309"

EDGE = "#334155"

def _frame(ax, xlim, ylim, title=None):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10, color=MUTED, pad=8)

def _box(ax, xy, text, w=2.6, h=0.9, fc=FILL, ec=EDGE, fs=9, bold=False):
    x, y = xy
    ax.add_patch(FancyBboxPatch(
        (x - w / 2, y - h / 2), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        linewidth=1.3, facecolor=fc, edgecolor=ec, zorder=2))
    ax.text(x, y, text, ha="center", va="center", fontsize=fs, color=INK,
            zorder=3, fontweight="bold" if bold else "normal")
    return xy

def _diamond(ax, xy, text, w=3.0, h=1.5, fc="#fff7ed", ec=WARM, fs=9):
    x, y = xy
    ax.add_patch(Polygon(
        [(x, y + h / 2), (x + w / 2, y), (x, y - h / 2), (x - w / 2, y)],
        closed=True, linewidth=1.3, facecolor=fc, edgecolor=ec, zorder=2))
    ax.text(x, y, text, ha="center", va="center", fontsize=fs, color=INK, zorder=3)
    return xy

def _dot(ax, xy, r=0.09):
    ax.add_patch(Circle(xy, r, facecolor=EDGE, edgecolor=EDGE, zorder=4))
    return xy

def _arrow(ax, pts, label=None, label_at=0.5, label_off=(0.0, 0.18),
           color=EDGE, ha="center"):
    """Poly-line arrow through `pts` (elbow routing), head on the last segment."""
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    ax.add_line(Line2D(xs[:-1] + [xs[-1]], ys[:-1] + [ys[-1]],
                       color=color, linewidth=1.3, zorder=1,
                       solid_capstyle="round"))
    ax.annotate("", xy=pts[-1], xytext=pts[-2],
                arrowprops=dict(arrowstyle="-|>", color=color, linewidth=1.3,
                                shrinkA=0, shrinkB=0), zorder=1)
    if label:
        i = max(0, min(len(pts) - 2, int(label_at * (len(pts) - 1))))
        mx = (pts[i][0] + pts[i + 1][0]) / 2 + label_off[0]
        my = (pts[i][1] + pts[i + 1][1]) / 2 + label_off[1]
        ax.text(mx, my, label, fontsize=8, color=MUTED, ha=ha, va="center")

def _line(ax, pts, color=EDGE):
    """Poly-line with no arrowhead — for merging branches into a shared rail."""
    ax.add_line(Line2D([p[0] for p in pts], [p[1] for p in pts], color=color,
                       linewidth=1.3, zorder=1, solid_capstyle="round"))

def _cellgrid(ax, values, origin=(0, 0), cw=1.0, ch=1.0, fs=13, fc="white"):
    """A row/table of boxed cells; `values` is a list of rows."""
    x0, y0 = origin
    for r, row in enumerate(values):
        for c, v in enumerate(row):
            x = x0 + c * cw
            y = y0 - r * ch
            ax.add_patch(plt.Rectangle((x, y - ch), cw, ch, facecolor=fc,
                                       edgecolor=EDGE, linewidth=1.2, zorder=2))
            ax.text(x + cw / 2, y - ch / 2, str(v), ha="center", va="center",
                    fontsize=fs, color=INK, family="monospace", zorder=3)

def _mockwindow(ax, w, h, title, body, titlebar="#d7dde5", face="#ffffff",
                fs=9, textcolor=INK):
    """A framed window with a title bar and monospaced body lines."""
    ax.add_patch(plt.Rectangle((0, 0), w, h, facecolor=face, edgecolor=EDGE,
                               linewidth=1.2, zorder=1))
    ax.add_patch(plt.Rectangle((0, h - 0.55), w, 0.55, facecolor=titlebar,
                               edgecolor=EDGE, linewidth=1.2, zorder=2))
    ax.text(0.2, h - 0.28, title, fontsize=9, va="center", color=INK, zorder=3)
    y = h - 1.05
    for line, colour in body:
        ax.text(0.25, y, line, fontsize=fs, va="center", family="monospace",
                color=colour or textcolor, zorder=3)
        y -= 0.5

def _index_grid(ax, items, top_label, side_label, fs=13):
    n = len(items)
    _cellgrid(ax, [items], origin=(0, 1), fs=fs)
    for i in range(n):
        ax.text(i + 0.5, 1.25, str(i), ha="center", va="bottom", fontsize=10,
                color=ACCENT)
    if top_label:
        ax.text(-0.25, 1.3, top_label, ha="right", va="bottom", fontsize=9,
                color=ACCENT)
    if side_label:
        ax.text(-0.25, 0.5, side_label, ha="right", va="center", fontsize=9,
                color=ACCENT)
    _frame(ax, (-6.4, n + 0.4), (-0.4, 2.0))

def _double_diamond(ax, stage=None):
    names = ["Understand", "Design", "Implement", "Evaluate"]
    # two diamonds: centres at x=2.6 and x=7.8, half-width 2.6, half-height 2.0
    for d, cx in enumerate((2.6, 7.8)):
        left, right, top, bot = cx - 2.6, cx + 2.6, 2.0, -2.0
        for half in (0, 1):
            name = names[2 * d + half]
            tri = ([(left, 0), (cx, top), (cx, bot)] if half == 0
                   else [(cx, top), (right, 0), (cx, bot)])
            on = (name == stage)
            ax.add_patch(Polygon(tri, closed=True, zorder=1,
                                 facecolor="#b9bfc7" if on else "#eceef1",
                                 edgecolor="none"))
            tx = cx - 1.3 if half == 0 else cx + 1.3
            ax.text(tx, 0, name, ha="center", va="center", fontsize=10,
                    color=INK if on else MUTED,
                    fontweight="bold" if on else "normal", zorder=3)
        ax.add_patch(Polygon([(left, 0), (cx, top), (right, 0), (cx, bot)],
                             closed=True, facecolor="none", edgecolor=INK,
                             linewidth=2.2, zorder=2))
        ax.plot([cx, cx], [top, bot], color=INK, linewidth=1.0, zorder=2)
    for x, y in ((0.0, 0.0), (5.2, 0.0), (10.4, 0.0)):
        ax.add_patch(Circle((x, y), 0.22, facecolor="#c9ced6", edgecolor=INK,
                            linewidth=1.2, zorder=4))
    ax.plot([-1.5, -0.22], [0, 0], color=INK, linewidth=1.6, zorder=2)
    ax.plot([10.62, 11.9], [0, 0], color=INK, linewidth=1.6, zorder=2)
    ax.plot([5.2, 5.2], [-0.22, -3.1], color=INK, linewidth=1.6, zorder=2)
    ax.text(-1.7, 0, "Problem", ha="right", va="center", fontsize=10)
    ax.text(12.1, 0, "Program", ha="left", va="center", fontsize=10)
    ax.text(5.2, -3.35, "Specification", ha="center", va="top", fontsize=10)
    _frame(ax, (-4.2, 14.2), (-4.2, 2.6))

def draw_turtle_rectangle_sides(ax):
    """Reproduces turtle_chap_modules.png — forward(50), left(90), forward(30)."""
    _mockwindow(ax, 8.0, 6.0, "Python Turtle Graphics", [])
    ax.plot([2.6, 5.1], [2.6, 2.6], color=INK, linewidth=1.4, zorder=3)
    ax.plot([5.1, 5.1], [2.6, 4.1], color=INK, linewidth=1.4, zorder=3)
    ax.annotate("", xy=(5.1, 4.35), xytext=(5.1, 4.0),
                arrowprops=dict(arrowstyle="-|>", color=INK, linewidth=1.4),
                zorder=3)
    _frame(ax, (-0.3, 8.3), (-0.3, 6.3))

def draw_turtle_five_circles(ax):
    """Reproduces turtle_chap_9.png — draw_circles(5)."""
    _mockwindow(ax, 9.0, 6.0, "Python Turtle Graphics", [])
    for k in range(5):
        ax.add_patch(Circle((2.3 + k * 1.15, 2.9), 0.6, facecolor="none",
                            edgecolor=INK, linewidth=1.3, zorder=3))
    ax.annotate("", xy=(7.15, 2.25), xytext=(6.75, 2.4),
                arrowprops=dict(arrowstyle="-|>", color=INK, linewidth=1.4),
                zorder=4)
    _frame(ax, (-0.3, 9.3), (-0.3, 6.3))

# ---------------------------------------------------------------- runtime ---
_SIZES = {'turtle_rectangle_sides': (5.6, 4.4), 'turtle_five_circles': (6.2, 4.4)}
_WIDTHS = {}
_FIGURES = {}


def _render(name):
    fig, ax = plt.subplots(figsize=_SIZES.get(name, (6.4, 4.4)), dpi=110)
    globals()["draw_" + name](ax)
    fig.tight_layout(pad=0.3)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return buf.getvalue()


def show(name, width=None):
    """Display one of this chapter's figures."""
    display(Image(_FIGURES[name], width=width or _WIDTHS.get(name, 560)))


for _n in ['turtle_rectangle_sides', 'turtle_five_circles']:
    _FIGURES[_n] = _render(_n)

print("Setup complete \u2014 2 figure(s) ready.")


In another chapter we talked about giving names to algorithms, as well as algorithms receiving data as input, and generating new data as output.  In Python, such named algorithms are implemented as *functions*.  Functions allow us to give a name to a block of Python code.  If an algorithm is implemented as a function in Python, we can run the algorithm by using the function's name in a Python program.  This is called *calling* the function.

another section of this chapter introduces functions as a method of abstraction, and their relationship to algorithms (Chapter ).
Later earlier we will introduce the Python syntax for calling functions, as well as some of the built-in functions available in Python.

### Functions and Abstraction

Functions are an example of abstraction.  They allow us to refer to a block of Python code by name, and ask for that code to be executed.  The code within a function can be executed without knowing what it is or how it works.  Let's talk about a function that you've already seen:  the `print()` function.  We introduced this function earlier and used it to output the values of expressions to the console, but we didn't reveal until now that it is a function!

Recall from another chapter that algorithms can have input.  Thus, it shouldn't surprise you that functions can have input, and indeed most functions require input.  Input to functions in Python are called *arguments*.  When we used the `print` function before, we provided expressions as arguments, and their values were output to the console.

We also said in another chapter that algorithms can have output.  Python functions can have output too!  When a Python function produces output, we say that the function *returns* a value.  The *return value* of a Python function is the output value that was produced.

In this way, we can use functions by providing input values (in the form of Python expressions), and receiving back output (in the form of return values).  This is a nice abstraction because we can send data to the function, the function executes, and produces its output, and we don't need to know **how** that output is arrived at.  All we need to know is **what** a function does, what inputs it requires, and what it returns as a result.

### Calling Functions

In Python we invoke the algorithm inside of a function by *calling* the function.  To call a function, we write the name of the function followed immediately by a pair of parentheses.  The *arguments*  to the function (inputs!) are given as a comma-separated list within the parentheses.  We illustrate this with one of the examples of the `print` function we saw in Section :

In [ ]:
a = 8
b = 5
print("The remainder after dividing", a, "by", b, "is:", 8 % 5)

The parts of a function call are:

- the **function name**;
- the **parentheses** enclosing the list of arguments;
- the **arguments** (inputs) to the function — notice how each argument is itself an expression; and
- the **commas** separating the arguments in the argument list.

All function calls have the same general format and look like this:

> *A fragment for illustration — it is not complete enough to run.*
```python
function_name(argument1, argument2, argument3, ... )
```

The `print`  function is a bit special in that it can accept any number of arguments.  Most functions are defined to have a fixed number of arguments for providing specific inputs.  The `len`  function in Python is an example; it accepts exactly one argument.   The `len` function can be used to find out how many characters are in a string.  You provide a string that you want to know the length of as the argument, like this:

In [ ]:
S = "No, I am your father!"
len(S) 
len("No.  No, that's not true.  That's impossible!")

Here we have two calls to the function `len`.  The first one returns the length of the string referred to by `S`, which is 21.  The second call returns the length of the string literal `"No.  No, that's not true.  That's impossible!"`, which happens to be 45.  In the next section we will discuss how to obtain and use the value returned by a function.

#### Functions as Expressions: Obtaining/Using a Function's Return Value

Function calls are expressions.  Like all other expressions they have a value.  The value of a function call is the return value of the function!  Remember, in interactive mode, if you enter an expression, Python prints out the value of the expression.  So here's what happens when we enter the expressions in the listing from the previous section:

In [ ]:
S = "No, I am your father!"
len(S) 
len("No.  No, that's not true.  That's impossible!")

Python prints out the return values of the function calls because the function calls are expressions whose value is the return value of the function call.  Since the value of a function call is its return value, you can use a function call wherever we can use an expression! Thus, function calls can be used...

- as operands of operators:

In [ ]:
len(S) + len("No.  No, that's not true.  That's impossible!")

- as values in assignment statements (give a name to the return value of a function):

In [ ]:
L = len(S)  
print(L)

The return value of `len` is 21, which gets assigned the name `L`.  Since `L` now refers to the value 21, the `print` function call outputs `21` to the console.
- as arguments to other functions:

In [ ]:
print(len(S), len("Search your feelings!"))

The return values of the `len` function are the arguments to the `print` function.  The two calls `len` happen first, and their return values are used as arguments to `print`.  Some people call this a *nested* function call, because a call to one function (`len`) is being made as part of a call to another function (`print`).  When nested function calls are used, the calls are made in order from inner-most to outer-most.

#### Calling Functions with No Arguments

It is possible for a function to be defined to have no arguments.  This means that the function doesn't have any input.  To call a function with no inputs, you simply leave the space between the parentheses empty.  We'll see an example of this later because right now we don't know any functions that require no arguments, and there aren't any interesting built-in Python functions that require zero arguments to use as an example.

#### Functions That Do Not Return a Value: Procedures

If a function has no output and does not return a meaningful value, then value of the function call is the special value `None`.  This means that, strictly speaking, it is not possible to have a function that returns nothing, because `None` is a value!

An example is the `print` function.  The `print` function doesn't need to return a meaningful value because it doesn't compute any new values, it just **does** something, namely, print its arguments to the console.  The following example proves that `print` returns the value `None`:

In [ ]:
x = print(42)
print(x)
x
print(y)

The first line gives the name `x` to the return value of `print(42)`.  Then `print(x)` proves that the value referred to by `x` is `None`.  So `print` returned `None`.  The next two lines show that there is a difference between the value `None`, and no value at all.  Typing just `x` is fine, because x refers to the value `None`.  But typing `y` results in a `NameError` because y refers to no value at all!  The variable `y` was never assigned to a value.

In mathematics, a function, by definition, always has a value.  Thus, functions in a programming language that do not return a value are sometimes referred to as a *procedure* since they are not, in the strict mathematical sense, functions.

#### `type` Functions

We have already seen one example of a **function call**:

In [ ]:
type(42)

The name of the function is `type`.  The expression in parentheses
is  the **argument** of the function.  The result, for this
function, is the type of the argument.

Python provides functions that convert values
from one type to another.  The `int` function takes any value and
converts it to an integer, if it can, or complains otherwise:

In [ ]:
int('32')
int('Hello')

`int` can convert floating-point values to integers, but it
doesn't round off; it chops off the fraction part:

In [ ]:
int(3.99999)
int(-2.3)

`float` converts integers and strings to floating-point
numbers:

In [ ]:
float(32)
float('3.14159')

Finally, `str` converts its argument to a string:

In [ ]:
str(32)
str(3.14159)

### Flow of execution

To ensure that a function is defined before its first use,
you have to know the order statements run in, which is
called the **flow of execution**.

Execution always begins at the first statement of the program.
Statements are run one at a time, in order from top to bottom.

Function definitions do not alter the flow of execution of the
program, but remember that statements inside the function don't
run until the function is called.

A function call is like a detour in the flow of execution. Instead of
going to the next statement, the flow jumps to the body of
the function, runs the statements there, and then comes back
to pick up where it left off.

That sounds simple enough, until you remember that one function can
call another.  While in the middle of one function, the program might
have to run the statements in another function.  Then, while
running that new function, the program might have to run yet
another function!

Fortunately, Python is good at keeping track of where it is, so each
time a function completes, the program picks up where it left off in
the function that called it.  When it gets to the end of the program,
it terminates.

In summary, when you read a program, you
don't always want to read from top to bottom.  Sometimes it makes
more sense if you follow the flow of execution.

The ability to create your own functions and create abstractions of your own algorithms is a tremendously powerful feature in any programming language.  The main reasons for writing your own functions are abstraction and decomposition of large programs into manageable pieces.  We can give names to our algorithms and abstract away their details by writing them as functions.
The purpose of this section is to learn how to do this in Python.

### Defining Functions and Parameters: The `def` Keyword

In Python, functions are defined by writing the keyword `def`.  A *keyword* is a name we give to words in a programming language that have special meaning.  Generally, keywords cannot be used as variable or function names, because Python will think you mean something else.  The best way to see how this works is with some examples.

#### Functions that Perform Simple Subtasks

The simplest form of function is one that takes no arguments as input (i.e. a procedure) and returns no value.  In another section we saw how to ask the user to enter their name, and then respond with a greeting.  Suppose this is an algorithm that we want to perform frequently.  We can create an abstraction of this algorithm by writing a function called `introductions` that performs the algorithm when we call it, allowing us the luxury of not having to remember how it works.  Here's what that would look like:

In [ ]:
def introductions():
	x = input("Please enter your name: ")  
	print("Hello,", x) 

Let's break down what's happening here.  The first line uses the keyword `def` to define a function called `introductions`.  The name of a function in a function definition must be followed by a pair of parentheses, then a colon.  Notice how the rest of the lines are indented.  In Chapter , we called this a *block* .  Just like in our pseudocode, we group statements together in blocks by indenting them.  All of the Python code that is part of a function has to be in the same block, so it has to be indented, and be indented by **exactly** the same amount or Python will complain thinking that some lines are not part of your function even when you want them to be.

> ⚠️ **This code is deliberately incorrect** — it illustrates a mistake.
```python
## Wrong indentation; Python will think that the print function
## is not part of the function, and will just execute it.
def introductions():
	x = input("Please enter your name: ")  
print("Hello,", x) 

## Wrong indentation; Python will issue an error here because the
## indentation is inconsistent, and it can't figure out 
## whether the print function call is part of the function or not.
def introductions():
	x = input("Please enter your name: ")  
  print("Hello,", x)
```

Indentation has **specific** meaning to Python.  Python is not like other languages where indentation is only cosmetic.  Indentation is used to indicate blocks which specify program structure.

Now, the correctly indented function definition doesn't actually do anything other than define the function.  Like any function, the code it contains only gets executed when it is **called** :

In [ ]:
# ▶ Interactive: this cell waits for you to type something.
# Run it yourself - "Run All" skips it so the rest of the chapter still works.
# defines the function only:
def introductions():
	x = input("Please enter your name: ")  
	print("Hello,", x) 

# this function call actually calls the function, 
# which executes its code.
introductions()   

#### Functions that Accept Arguments

We have already shown you functions that accept input using arguments.  To define your own function that accepts arguments, you can add a comma-separated list of variable names between the parentheses.  These are called the function's *parameters*.  Parameters are the variables that the function uses to refer to the arguments it is given in a call to that function.  Here we have modified our `introductions` function to have a parameter called `greeting`.

In [ ]:
# ▶ Interactive: this cell waits for you to type something.
# Run it yourself - "Run All" skips it so the rest of the chapter still works.
# defines the function only:
def introductions(greeting):
	print(greeting)
	x = input("Please enter your name: ")  
	print("Hello,", x) 

# this function call actually calls the function, 
# which executes its code.
introductions("Welcome to my Python program!")   

This defines a function that takes one argument when you call it; the last line of the program shows the function being called with a string as an argument.  The function, internally, refers to that argument by the variable name `greeting`.  We have written the function so that it assumes that the greeting is a string, which it prints out prior to asking for the user to enter their name.  If we execute the above Python program, this is what we will see:

Output:
```text
Welcome to my Python program!
Please enter your name: Chris
Hello, Chris
```

The bright red text was entered by the user.  The string `'Welcome to my Python program!'` was used as an argument to the function `'introductions'`.  The argument was then assigned the parameter name `greeting`, so that within the function, the variable `greeting` refers to the string `'Welcome to my Python program!'`, which is why this is the output we get when the function executes `print(greeting)`.

A function can have any number of parameters.  Each parameter you add corresponds to an argument that must be provided when the function is called.  Parameters are assigned to refer to arguments in the same order they are given.  For example, if a function was defined in this way:

> *A fragment for illustration — it is not complete enough to run.*
```python
def functionWithManyArguments(a, b, c, d):
	# code for function would go here

X = False	
functionWithManyArguments(42.0, "Good Morning", 17, X)
```

then the function call would assign the parameter name `a` to refer to the argument `42.0`, the parameter name `b` to refer to the argument `'Good Morning'`, the parameter name `c` to refer to the argument `17`, and the parameter name `d` to refer to the parameter `X`.  When an argument is a variable, like `X` in this example, the parameter name is assigned to refer to the value that the argument refers to, so actually, the parameter `d` ends up referring to the value `False`.

Parameters are variables that get their values from the arguments in a function call; Python does the assignment of parameter to argument behind the scenes, but it is the same assignment we studied in Chapter .  A key point to understand is that a parameter always refers to a value that was created outside the function. The parameter is simply the function's name for it.

#### Returning a Value

We've seen how to use parameters to define a function that accepts inputs (arguments).  We've also seen built-in Python functions that return a value.  So how do we have one of our own functions return a value?

The answer is pretty simple:  Write the keyword `return`, followed by an expression.  The value of the expression becomes the return value for the function, and the value of the call that invoked the function.  For example, we could modify our `introduction` function to return the name that the user entered, so that it can be used by the caller for future reference:

In [ ]:
# ▶ Interactive: this cell waits for you to type something.
# Run it yourself - "Run All" skips it so the rest of the chapter still works.
# defines the function only:
def introductions(greeting):
	print(greeting)
	x = input("Please enter your name: ")  
	print("Hello,", x) 
	return x

# this function call actually calls the function, 
# which executes its code.
username = introductions("Welcome to my Python program!")   

In this example, the return statement at the end of `introductions` causes the value referred to by `x` (the text the user entered) to be returned.  The execution of the program then resumes immediately after the function call to `introductions`, and since the name `username` was assigned to the return value of the function call, it now refers to the text that the user entered.  Once the function has returned, and execution has resumed after the function call, the variable `x` no longer exists.  Returning a value is one way of getting data out of a function.

We will see later that functions may have more than one return statement (Some are of the opinion that functions with more than one return statement is bad style!  Some believe otherwise.  We really aren't too worried about it.).  As soon as a return statement is executed, regardless of where it appears in the function, execution of the function immediately ceases (even if there are lines of code after it!), the value of the accompanying expression is returned, and execution continues from the line immediately after the call that invoked the function (or, in some cases, the line containing the function call continues executing, e.g. if the function call was part of a variable assignment, the assignment occurs after the function call returns).

#### Returning Nothing

If a function does not need to return a value, then simply do not include a return statement.  After the execution of the last line of the function, the value `None` will be returned by default.  As we noted before, such a function is sometimes called a *procedure*.

#### Defining Before Calling

Python functions must be defined before they are called.  Thus a function definition must appear in a file prior to any calls to that function.  In the following code, the first call to `introductions` would fail and cause a `NameError`.  But the second call to `introductions` would work fine because its definition appears first.

In [ ]:
# this fails -- function called before definition
username = introductions('Welcome to my Python program!')   

def introductions(greeting):
	print(greeting)
	x = input("Please enter your name: ")  
	print("Hello,", x) 
	return x

# this is fine - function called after definition.
username = introductions("Welcome to my Python program!")   

#### Summary of functions

If we want to write a function that has inputs, we need to give it parameters.  Parameters are the variable names that are used within the function to refer to the values of a function's arguments; they are given in the function's **definition**.  Arguments are the input values for the function; they are provided when the function is **called**.

A function can be instructed to return a value (i.e. produce an output!) using the `return` keyword.  The code for the function must be indented in a *block*.  Indentation has semantic meaning in Python and must be used properly and with care.

### Variable Scope

The *scope* of a variable refers to the parts of the program in which a variable exists after it is assigned to a value.  This sounds complicated but it's actually quite straightforward.  Variables defined within a function only exist within that function.  Variables defined outside any function do not exist within any functions.  Here's an example:

In [ ]:
def fireball_damage():
	damage = 30
	return damage

damage = 0
D = fireball_damage()	
print(damage)

In this example, you might expect the value 30 to be printed.  In fact, the value 0 is printed.  To see why this is, you must realize that we have two **different** variables called `damage` in this program.  The `damage` variable defined in the function `fireball_damage` only exists within the function -- we say that it is a *local* variable.  Likewise, the `damage` variable defined outside of the function only exists outside of the function.  It's scope is said to be *global*.  Thus, the `fireball_damage` function is changing only what the local variable `damage` refers to, not what the `damage` variable defined outside of the function refers to.  Moreover, two different functions can use the same variable name, but they are, in fact, completely different and unrelated variables!

### Console I/O vs Function I/O

In computer science, we use the words *input* and *output* a lot, and we use them to mean different things.  For example, the inputs and outputs of a **function** are different and distinct from *console input* and *console output*.  Function inputs are the arguments of a function call assigned to function parameters; function outputs are returned from data created within a function.   Console inputs are read from the keyboard; console outputs are printed to the screen rather than being sent to another part of a program.

When reading instructions it is important to be able to distinguish these forms of input and output. If you are asked to write a function that "takes" something as input, this means the function should take an argument via a parameter.  If you are asked to write a function that "reads from the console", or "asks the user" for some data, this means that the function should perform console input to get this data by calling the `input` function.  If you are asked to write a function that "outputs", "prints", or "displays" some data, then this should be done with console output by calling the `print` function.  If you are asked to write a function that "returns" some data, then this should be done using the `return` keyword.

### Documenting Function Behaviour

Python does not restrict the data type of function arguments, so you can pass an argument of any type as the argument for any parameter of a function.  But usually functions expect arguments to be of a certain data type.  How do we communicate these expectations to the programmer who wants to call the function?

When we write a function we should *document* what its inputs and outputs are.  We can do this by writing *docstrings* in our program.  Docstrings are a way of describing what the function does, what each parameter is for, the expected data type of the argument to that parameter, and what the function returns (if anything).  Here's how we would do this for our `introductions` function:

In [ ]:
# ▶ Interactive: this cell waits for you to type something.
# Run it yourself - "Run All" skips it so the rest of the chapter still works.
def introductions(greeting):
	"""
Greet the user and asks them for their name.

greeting: A string containing a message to greet the user
Returns: The name entered by the user.
	"""
	print(greeting)
	x = input("Please enter your name: ")  
	print("Hello,", x) 
	return x

# this function call actually calls the function, 
# which executes its code.
username = introductions("Welcome to my Python program!")   

The docstring is enclosed in triple double-quotes and is indented with the rest of the block of code for the function. (The first set of triple double-quotes must be indented, but the rest of the docstring need not be because Python interprets the entire docstring as a single line of text.)  The triple double-quotes are how you specify a multi-line string literal in Python.  The docstring should contain a brief one-line description of what the function does, followed by a list of parameters, and what they are for, followed by a description of what the function returns.  There are no particular formatting requirements for the contents of the docstring, but you should strive for something similar to the above.

If a function has a docstring, you can view it by typing `print(*functionname*.__doc__)`.

In [ ]:
print(introductions.__doc__)

It also works for built-in functions, like `pow` or `max`:

In [ ]:
print(pow.__doc__)

### Generalization

*Generalization* of functions (or algorithms) is the process of modifying a function/algorithm that solves a specific problem so that it can solve a wider range of problems, or a larger number of instances of the same problem.  Let's consider that we must ask the user to input a set of grades from different courses and calculate the average grade. We could write a function to do this:

In [ ]:
def average_grades(num_grades):
    average = 0  # initialize the average at 0
    
    # use a loop to repeat the input process
    for i in range(1, num_grades+1): 
        # ask for the grade
        average += int(input(f"Enter grade {i}: "))  
    
    # calculate the average and return
    average = int(average/num_grades)
    return average

This function takes as input arguments the number of grades we want the user to input. It then uses a `for` to prompt the input that many times. Finally, it calculates and returns the average. Now we can call this function whenever we need to know the average of grades entered by the user, without having to remember how that is calculated.

There could be other parts of the program, or other programs, where averaging a set of numeric inputs could be useful. These inputs may not necessarily be grades, they could represent the population of towns and cities on PEI (see Chapter ), precipitation values or temperature measurements. We could generalize this function to apply to those situations by asking the user to enter a value in general and not specifically asking for a grade. It would also be convenient to treat the input as a floating-point number instead of restricting it to be an integer and returning a floating-point average (we can convert to an integer outside the function if necessary). The generalized function would be:

In [ ]:
def average_values(num_values):
    average = 0  # initialize the average at 0
    
    # use a loop to repeat the input process
    for i in range(1, num_values+1): 
        # ask for the grade
        average += float(input(f"Enter numeric value {i}: "))  
    
    # calculate the average and return
    average = int(average/num_values)
    return average

These may seem like small changes, but now we have a function that can solve the same problem in a much wider range of situations!  Can you think of how we might generalize this further (see the footnote for the answer!)? (We may use a sentinel loop in case we don't know upfront how many values will be input.)

The more general a function is, the more re-usable it is.  The more often we can re-use existing code that has been tested and proven to work, rather than write new code, the less likely we are to introduce errors into programs.

Of course, there is a limit to this.  Functions that do too much or too many different things are actually bad.  Thus, generalization must be tempered by another concept called *cohesion* which we discuss in the next section.

### Cohesion

In software design, the term *cohesion* refers to the idea that code that is grouped together should have something in common.  In terms of writing functions, functions that perform one task and one task only are said to have high cohesion.  Functions with high cohesion are preferred because they increase the reusability and maintainability of software components.   Our function from another section that computes how many users can stream Netflux on an internet connection of a certain speed has high cohesion because it performs a single, well-defined task.

An example of low cohesion would be a function that, say, not only computed the number of users that can simultaneously stream on a connection but also includes a parameter that changes the user's streaming quality (e.g. standard or high definition).  These are two different tasks that are entirely independent, and should be implemented in separate functions.

### Modules:  What Are They and Why Do We Need Them?

Modules are files that contain function and object definitions that add additional, and much more powerful features to Python.  The basic Python language provides only very fundamental building blocks for programs.  Modules are a way for people to share functions and objects that they have written so that other people can use them in their own programs.  In this respect, modules are similar to the *libraries* that are used by other programming languages such as C++ and Java.  Viewed another way, modules contain *abstractions* of algorithms that we can use in our own programs without having to understand how they work.

### How to Use Modules

Modules are stored in separate files from our own programs.  Thus, in order to use the functions and objects defined by a module, we have to tell our own program to look in those files and read those definitions.  We do this using a Python keyword called `import`.

Perhaps you noticed that Python, by default, doesn't seem to be able to compute common mathematical functions like logarithms and finding the square root of a number, or trigonometry functions such as sine, cosine, and tangent.  The reason for this is that these functions are instead defined in a module called `math`.  If we want to use the functions in the `math` module, we need to `import` them from the `math` module   into our program.  For example:

In [ ]:
log10(1000)   # this won't work: there is no such function yet

In [ ]:
import math as m   # read the definition of the log function
m.log10(1000)      # now we can compute base-10 logarithms!

Our first call to `log10` fails, because Python does not have a built-in function called `log10`.  The command `import math as m` reads the function definitions from the `math` module and creates an object called `m` that contains the functions defined by the `math` module (i.e. the functions defined by `math` become methods of `m`).  Since the `log10` function is defined in the `math` module, the object `m` contains the `log10` method, and we can call it in the same way we call any method in an object using the dot notation we learned in Section .  As we can see, above, `m.log10(1000)` returns the correct value `3.0.`

Below is a program showing some more examples of using functions from the `math` module.

In [ ]:
import math as m

print(m.sqrt(7.5))     # display square root of 7.5
print(m.exp(5))        # display e to the power of 5
print(m.log2(256))     # display the base-2 logarithm of 256
angle = m.radians(90)  # convert 90 degrees to radians.
print(m.sin(angle))    # display the sine of 90 degrees.

Run this program in Python, and you'll see that it produces the output described.  If you're wondering why we used the `radians` function to convert the angle 90 degrees to radians, it's because the `sin` function requires that its argument be an angle in radians.

If you want to see a complete list of the functions in the `math` module, and documentation on how to use them, click on the following link, or copy it into your web browser:  <https://docs.python.org/3/library/math.html>.

> **Import Syntax**
>
> In general, the syntax for importing modules is:
> `import $x$ as $y$`
> $x$ must be the name of a module, and $y$ must be a valid variable name.  This creates an object called $y$ that contains, as methods, the functions defined in $x$.

### What Other Modules Are There?

In this course, we use a Python distribution called Anaconda.  Anaconda comes with a lot of  modules, too many to list here.  This is one of the great things about Python.  There are so many modules available for it, that there is probably a module to either do, or help you do almost anything you can think of.  It is also possible to obtain and use modules that do not come with Anaconda.  These take the form of Python program  files (files with a `.py` extension) that can be placed in the same folder as your program, and then imported.  It is also possible to write your own modules. (A module is just a `.py) file that contain only function and/or object definitions.  You can write your own group of related functions and use them as a module!`

In this course we will be using several different modules that come with Anaconda, including ones that can:

- read, write, modify, and display image files;
- plot line and bar graphs; and
- draw graphics to the screen.

#### The Matplotlib Module

Let's look at one more example, from `matplotlib` — the same module that draws the figures in this book.  As well as charts, `matplotlib` can read and display images.  The following Python code reads a JPEG image file and displays it on the screen:

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib import cbook

# matplotlib ships this photograph, so it works offline
with cbook.get_sample_data("grace_hopper.jpg") as f:
    im = mpimg.imread(f)

plt.imshow(im)
plt.axis("off")
plt.show()

The first two lines read definitions out of the `matplotlib` module.  `cbook.get_sample_data` finds a photograph that is shipped inside `matplotlib` itself, so this example works without downloading anything.  `mpimg.imread` reads that image file and returns an object holding the image data (the data comes back as an *array* — we will meet arrays in a later chapter).  `plt.imshow` adds the image to the figure being built, `plt.axis("off")` hides the x- and y-axes that would otherwise be drawn around it, and `plt.show` displays the result.
This image from the public domain was download from <pixabay.com>.
Look at that cute parrot.  It's gorgeous!  Now think about how much is actually going on behind the scenes in those few lines of code.  The file on the disk has to be opened, the data has to be decompressed, decoded, and loaded into memory, and then it has to be sent to the display hardware.  These are all quite complex operations with many many steps.  But thanks to abstraction, we accomplished all that with just three simple method calls to `imread`, `imshow`, and `show`.

#### The Turtle Module

Let's now take a look at a very fun module: the turtle module. This module enables users to create pictures and shapes. The onscreen pen that you use for drawing is called the turtle and this is what gives the module its name. This module helps new programmers get a feel for what programming with Python is like in a fun and interactive way.

Here we have a few lines of Python that create a new turtle and draw two sides of a rectangle. We have called our first turtle *leo*, but we can choose any other name we like (maybe *raphael* or *donatello*).

> *Turtle graphics need a graphics window. Run this one in `CS1910_Computational-Thinking_Workbook.ipynb`, which sets up `ColabTurtlePlus` for you. The result is shown below.*
```python
import turtle               # Allows us to use turtles
window = turtle.Screen()    # Creates a playground for turtles
leo = turtle.Turtle()       # Creates a turtle object and assigns it

leo.forward(50)             # Tells leo to move forward by 50 units
leo.left(90)                # Tells leo to turn by 90 degrees
leo.forward(30)             # Completes the second side

window.mainloop()           # Waits for user to close the window
```

The output of this code are two orthogonal lines, representing two sides of a rectangle.

In [ ]:
show("turtle_rectangle_sides")

#### Drawing circles

We can also draw more sophisticated things with the turtle module, for example, we can use a counting while-loop to write a function that draws a row of $n$ circles on the screen:

In [ ]:
show("turtle_rectangle_sides")

> *As above — run this in the lecture workbook. The result is shown below.*
```python
import turtle as turtle

def draw_circles(n):
    circles_drawn = 0  # number of circles drawn so far
    leo = turtle.Turtle()
    window = turtle.Screen()
    while circles_drawn < n:  # while we haven't drawn n circles
        leo.goto(circles_drawn * 50, 0)  # move the turtle

        leo.down()  # put the turtle's pen down
        leo.circle(20)  # draw a circle of radius 20 pixels
        leo.up()  # pick up the turtle's pen

        # add 1 to the number of circles drawn
        circles_drawn = circles_drawn + 1
    window.mainloop()
```

In this example the `draw_circles` function has a parameter that determines the number of circles to draw.  The  variable `circles_drawn` acts as a counter that keeps track of how many circles we've drawn, and that the while-loop's condition `circles_drawn < n` causes the while-loop's block to execute until we have drawn exactly `n` circles. If we were to call the `draw_circles` function with an argument of `5`, like this: `draw_circles(5)` then we'd see the following output consisting of five circles in a row:

In [ ]:
show("turtle_five_circles")

### Finding Module Documentation

At this point, I am sure you are wondering how one finds out about modules, and how to use them.  The short answer is:  internet search.  For example, a search for "skimage display image" returns us a link to the documentation for the `skimage.io` module.  That link is here:  <http://scikit-image.org/docs/dev/api/skimage.io.html>. For the turtle module you can find the documentation here: <http://docs.python.org/3.3/library/turtle.html>

While it can sometimes be hard to find documentation for modules, rest assured that, for this course, we'll always tell you how to use a module function or object that we expect you to use, or at least tell you exactly where the documentation is.

In [ ]:
show("turtle_five_circles")